# CubbyLLM: the Triton kernels, tested and timed on this GPU

The fused MinGRU recurrence (`cubbyllm/model/backbone/mingru_triton.py`, `CB_MINGRU_TRITON=1`) and the banded
flash attention (`window_attn_triton.py`, `CB_ATTN_TRITON=1`), each against what it replaces: their tests on the
card (values and gradients against the references, the mixers under their flags), then one layer's forward +
backward at the 450M's shapes (`validation/bench_kernels.py`) -- with flex attention at its default tiles and at
tiles for a 99 KB card beside them. No Drive, no checkpoint: ~5 minutes on any GPU.

In [ ]:
import os, subprocess, sys
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git pull -q --ff-only
!cd /content/CubbyLLM && git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, triton; print('torch', torch.__version__, '| triton', triton.__version__)

def stream(cmd, **env):
    p = subprocess.Popen(cmd, cwd='/content/CubbyLLM', env=dict(os.environ, **env), text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        if 'Warning' not in line and 'warn' not in line:
            print(line, end='')
    return p.wait()

In [ ]:
# --- the tests, on this card ---
stream([sys.executable, '-m', 'pytest', '-q', '-rs', '-p', 'no:cacheprovider',
        'tests/model/backbone/test_mingru_triton.py', 'tests/model/backbone/test_window_attn_triton.py'])

In [ ]:
# --- one layer, forward + backward, at the 450M's shapes (micro 32 x 1024 tokens; 8 on a card under 40 GB) ---
B = '32' if torch.cuda.get_device_properties(0).total_memory > 40 * 2**30 else '8'
stream([sys.executable, '-u', 'validation/bench_kernels.py'], CB_B=B, CB_OUT='/content/bench_kernels.json')